# Topic Segmentation Pipeline

Splits a transcript into topics using sentence embeddings and cosine similarity.

**Steps:** Split into sentences → Embed → Compare consecutive pairs → Find boundaries → Group into topics

## Setup

In [4]:
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt_tab', quiet=True)
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1: Split VibeVoice segments into sentences

VibeVoice returns segments by speaker turn, not by sentence.
We use NLTK to split each segment into individual sentences,
estimating timestamps based on position within the segment.

In [5]:
# Simulated VibeVoice output
segments = [
    {"start": 0.0, "end": 14.0, "speaker": 0,
     "content": "All right, so here we are, in front of the elephants. Cool thing about these guys is that they have really long trunks. And that's cool."},
    {"start": 14.0, "end": 16.2, "speaker": None, "content": "[Unintelligible Speech]"},
    {"start": 16.2, "end": 19.01, "speaker": 0,
     "content": "And that's pretty much all there is to say."}
]

all_sentences = []
for seg in segments:
    if seg["content"].startswith("["):  # skip [Music], [Unintelligible Speech], etc.
        continue
    sentences = sent_tokenize(seg["content"])
    seg_duration = seg["end"] - seg["start"]
    time_per_sentence = seg_duration / len(sentences) if sentences else 0
    for i, sentence in enumerate(sentences):
        all_sentences.append({
            "text": sentence,
            "start": round(seg["start"] + (i * time_per_sentence), 2),
            "end": round(seg["start"] + ((i + 1) * time_per_sentence), 2),
            "speaker": seg["speaker"],
        })

for s in all_sentences:
    print(f"[{s['start']:.1f}s - {s['end']:.1f}s] Speaker {s['speaker']}: {s['text']}")

[0.0s - 4.7s] Speaker 0: All right, so here we are, in front of the elephants.
[4.7s - 9.3s] Speaker 0: Cool thing about these guys is that they have really long trunks.
[9.3s - 14.0s] Speaker 0: And that's cool.
[16.2s - 19.0s] Speaker 0: And that's pretty much all there is to say.


## Steps 2-5: Full segmentation pipeline

Using a longer lecture transcript to demonstrate topic detection.
The algorithm: embed sentences → cosine similarity between consecutive pairs → percentile cutoff → group.

In [6]:
# Longer transcript with 3 distinct topics
lecture_sentences = [
    # Topic: What are neural networks
    "Today we're going to talk about neural networks.",
    "A neural network is a computing system inspired by the brain.",
    "It consists of layers of interconnected nodes called neurons.",
    "Each connection between neurons has a weight that gets adjusted during training.",
    # Topic: Training process
    "Now let's discuss how we actually train these networks.",
    "Training involves feeding data through the network and comparing the output to the expected result.",
    "The difference between the output and the expected result is called the loss.",
    "We use an algorithm called backpropagation to minimize this loss.",
    "Backpropagation computes gradients that tell us how to adjust each weight.",
    # Topic: Practical tips
    "Let me share some practical tips for training your models.",
    "Always start with a small learning rate and increase it gradually.",
    "Make sure to split your data into training and validation sets.",
    "Overfitting is a common problem where the model memorizes the training data.",
    "You can use techniques like dropout and data augmentation to prevent overfitting.",
]

# Step 2: Embed each sentence into a 384-dim vector
embeddings = model.encode(lecture_sentences)

# Step 3: Cosine similarity between consecutive sentences
similarities = []
for i in range(len(embeddings) - 1):
    # Get vectors for two consecutive sentences
    current = embeddings[i].reshape(1, -1)      # reshape (384,) to (1, 384)
    next_one = embeddings[i + 1].reshape(1, -1)

    # Compare them — returns [[score]]
    similarity_matrix = cosine_similarity(current, next_one)

    # Extract the single number
    score = similarity_matrix[0][0]
    similarities.append(score)

# Step 4: Percentile-based boundary detection
PERCENTILE = 15
cutoff = np.percentile(similarities, PERCENTILE)
boundaries = [i + 1 for i, s in enumerate(similarities) if s < cutoff]

# Step 5: Group sentences into topics
topics = []
start_idx = 0
for boundary in boundaries:
    topics.append(lecture_sentences[start_idx:boundary])
    start_idx = boundary
topics.append(lecture_sentences[start_idx:])

# Display results
print(f"Cutoff ({PERCENTILE}th percentile): {cutoff:.3f}")
print(f"Boundaries at sentences: {boundaries}")
print(f"Number of topics: {len(topics)}\n")

for i, topic in enumerate(topics):
    print(f"{'='*60}")
    print(f"TOPIC {i + 1} ({len(topic)} sentences)")
    print(f"{'='*60}")
    for sentence in topic:
        print(f"  {sentence}")
    print()

Cutoff (15th percentile): 0.383
Boundaries at sentences: [9, 12]
Number of topics: 3

TOPIC 1 (9 sentences)
  Today we're going to talk about neural networks.
  A neural network is a computing system inspired by the brain.
  It consists of layers of interconnected nodes called neurons.
  Each connection between neurons has a weight that gets adjusted during training.
  Now let's discuss how we actually train these networks.
  Training involves feeding data through the network and comparing the output to the expected result.
  The difference between the output and the expected result is called the loss.
  We use an algorithm called backpropagation to minimize this loss.
  Backpropagation computes gradients that tell us how to adjust each weight.

TOPIC 2 (3 sentences)
  Let me share some practical tips for training your models.
  Always start with a small learning rate and increase it gradually.
  Make sure to split your data into training and validation sets.

TOPIC 3 (2 sentences)
  O

## Experiment: Percentile tuning

Lower percentile = fewer topics (only the most obvious breaks).
Higher percentile = more topics (splits on smaller changes).

In [7]:
for pct in [10, 15, 20, 25, 30]:
    cutoff = np.percentile(similarities, pct)
    num_boundaries = sum(1 for s in similarities if s < cutoff)
    print(f"  Percentile {pct}% → cutoff {cutoff:.3f} → {num_boundaries + 1} topics")

  Percentile 10% → cutoff 0.321 → 3 topics
  Percentile 15% → cutoff 0.383 → 3 topics
  Percentile 20% → cutoff 0.405 → 4 topics
  Percentile 25% → cutoff 0.407 → 4 topics
  Percentile 30% → cutoff 0.412 → 5 topics
